## Route acquisition

Detour reranking starts with a route. `route_engine.RouteEngine` is the *head-agnostic*
route model of the demo: it imports no `ipywidgets` and no `ipyleaflet`, so it can be
reused as-is in another application. It holds the route state, calls the
[HERE Routing API][1], caches what it gets back, and derives the context sent to the
[HERE search & geocoding API][2].

A route is defined by a start, a stop and a corridor width in meters. Setting all three
is enough to retrieve the route:

[1]: https://www.here.com/docs/bundle/routing-api-v8-api-reference/page/index.html
[2]: https://docs.here.com/geocoding-and-search/reference

In [ ]:
from here_search_demo.auth import Credentials
from here_search_demo.route_engine import RouteEngine

credentials = Credentials()
routing_calls = []  # to count the Routing API requests

engine = RouteEngine(credentials=credentials, on_routing_request=lambda: routing_calls.append(1))
engine.set_route_start((48.57436, 7.71556))  # Strasbourg
engine.set_route_stop((45.75049, 4.84418))  # Lyon
engine.set_route_width(500)  # corridor width in meters

cached = await engine.update_route_attributes()
len(routing_calls), engine.route_summary_length, engine.waypoints_count

One single `/v8/routes` request was sent. `RouteEngine` delegates it to a
`routing_api.RoutingAPI` instance, asking for
`return=polyline,summary&spans=duration,dynamicSpeedInfo,length`, so that request returns
everything the demo needs: the geometry, the total length, and the per-span durations
used later to move the search center forward in time.

In [ ]:
list(cached.keys()), cached["spans"][0]

### The route cache

`RouteEngine` stores its results in a plain `dict` keyed by tuples. Two keys are written
by a single route acquisition:

* `("route", start_lat, start_lon, stop_lat, stop_lon)` — geometry, spans and length
* `("summary", start_lat, start_lon, stop_lat, stop_lon)` — `(duration, length, polyline)`

The second one is pre-populated on purpose: it is the exact key `DetourRanker` looks up
for its `at → stop` baseline leg. When the current position is still the route start, the
baseline is therefore free. Sharing that dict between the route engine and the detour
ranker is the single most effective way to reduce the Routing API bill:

In [ ]:
for key in engine._route_cache:
    print(key)

### Two polylines

The route is exposed as two [flexible polylines](https://github.com/heremaps/flexible-polyline):

* `route_flexpolyline` — the full geometry, used for rendering
* `search_flexpolyline` — the same geometry simplified to at most
  `RouteEngine.max_waypoints_count` points, used as the `route=` body of the search request

Search requests carrying a route are POSTed, so the polyline never hits a URL length limit,
but it is still worth keeping it small:

In [ ]:
RouteEngine.max_waypoints_count, len(engine.route_flexpolyline), len(engine.search_flexpolyline)

The simplification and the corridor buffer live in `widgets.route_geometry`. Despite the
module location, both functions are pure geometry (`shapely` only) and can be called
outside any widget:

In [ ]:
from here_search_demo.widgets.route_geometry import build_corridor, simplify_polyline

corridor = await build_corridor(cached["waypoints"], engine.width)
print("corridor bounds:", tuple(round(v, 4) for v in corridor.bounds))
print("simplified waypoints:", len(simplify_polyline(cached["waypoints"], 50)))

### Where is the driver, and where will they be?

`set_current_position()` places the car on the route. `set_mins_from_pos()` shifts the
search center *N* driving-minutes ahead of it, by walking the span durations returned with
the route. `search_at_position` returns the future position when one is set, the current
position otherwise: it is the `at=` of the search request **and** the origin of every
detour computation.

In [ ]:
engine.set_current_position((48.03529, 7.37121))  # near Colmar, already on the way
print("at now:      ", engine.search_at_position)

engine.set_mins_from_pos(5)
print("at in 5 min: ", engine.search_at_position)

### Searching along the route

`API.browse()`, `API.discover()` and `API.autosuggest()` all accept a `polyline` and a
`width`. When a polyline is given the request becomes a POST with a
`route=<polyline>;w=<width>` body, so the request is never limited by a URL length, and the
Search API only returns the places lying inside the corridor.

In [ ]:
from here_search_demo.api import API, HTTPSession

api = API(credentials=credentials)
session = HTTPSession()

latitude, longitude = engine.search_at_position
resp = await api.browse(
    session=session,
    latitude=latitude,
    longitude=longitude,
    categories=["100-1000"],  # restaurants
    polyline=engine.search_flexpolyline,
    width=engine.width,
    limit=20,
    lang="en",
)
print(resp.req.full)
print(resp.req.data[:60], "...")
resp.titles

Those restaurants are inside the corridor, but their order says nothing about the
time it takes to drive to them. The [next notebook](obm_7_detour.ipynb) turns it into a *driving time* order.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>